# Smart Sampa × roubos e furtos de celulares — EDA preliminar

Esta análise cruza a fotografia de câmeras por subprefeitura em setembro de 2025 com os BOs de roubo/furto de celulares ocorridos em 2025 e espacialmente atribuídos pelo GeoSampa.

**Importante:** é uma análise transversal e associativa, não uma estimativa causal de eficácia ou deterrência.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path('..')
data = pd.read_csv(ROOT / 'data/processed/analytical_subprefeituras_2025.csv')
summary = pd.read_csv(ROOT / 'data/processed/analysis_summary_2025.csv')
quartiles = pd.read_csv(ROOT / 'data/processed/analysis_quartiles_2025.csv')
data.shape

(32, 19)

## Cobertura espacial da SSP-SP

A análise territorial usa apenas a parcela dos BOs que possui coordenada válida e pôde ser associada a um polígono oficial. O arquivo `ssp_geocoding_quality_month.csv` preserva essa auditoria mês a mês.

In [2]:
summary.loc[summary['metrica'].isin([
    'bos_elegiveis_2025',
    'bos_com_coordenada_valida_2025',
    'bos_atribuidos_subprefeitura_2025',
    'pct_bos_atribuidos_subpref_total',
    'pct_spatial_join_subpref_entre_coords_validas',
])


                                         metrica      valor
2                             bos_elegiveis_2025  161145.00
3                 bos_com_coordenada_valida_2025  133051.00
4              bos_atribuidos_subprefeitura_2025  132933.00
7               pct_bos_atribuidos_subpref_total      82.49
8  pct_spatial_join_subpref_entre_coords_validas      99.91

## Associação espacial

A variável principal compara **câmeras por 10 mil habitantes** com **BOs geocodificados por 100 mil habitantes**. O nome das variáveis registra explicitamente que o denominador populacional é o Censo 2022.

In [3]:
data[['cameras_por_10_mil_hab_pop2022','celulares_geocod_por_100_mil_pop2022']].corr(method='pearson')

                                      cameras_por_10_mil_hab_pop2022  celulares_geocod_por_100_mil_pop2022
cameras_por_10_mil_hab_pop2022                              1.000000                              0.770441
celulares_geocod_por_100_mil_pop2022                        0.770441                              1.000000

In [4]:
data[['cameras_por_10_mil_hab_pop2022','celulares_geocod_por_100_mil_pop2022']].corr(method='spearman')

                                      cameras_por_10_mil_hab_pop2022  celulares_geocod_por_100_mil_pop2022
cameras_por_10_mil_hab_pop2022                              1.000000                              0.737903
celulares_geocod_por_100_mil_pop2022                        0.737903                              1.000000

![Dispersão entre cobertura de câmeras e BOs geocodificados](../reports/figures/cameras_vs_cellphones_percap_2025.svg)

## Quartis de cobertura

In [5]:
quartiles

  quartil_cameras_percap  n_subprefeituras  cameras_10k_mediana  celulares_100k_mediana  roubos_100k_mediana  furtos_100k_mediana
0               Q1_menor                 8                 9.34                  484.65               266.27               211.27
1                     Q2                 8                19.28                  639.81               334.20               318.70
2                     Q3                 8                27.10                  764.70               324.37               359.52
3               Q4_maior                 8                74.83                 2015.32               688.91              1380.18

![Mediana dos BOs geocodificados por quartil de câmeras](../reports/figures/camera_quartiles_vs_cellphones_2025.svg)

## Leitura metodológica

O resultado central é que a associação observada é positiva, não negativa. Isso **não** demonstra que câmeras aumentam crimes nem que sejam ineficazes. Uma explicação plausível é causalidade reversa/seleção: locais com maior circulação e criminalidade podem receber mais câmeras.

Para avaliar deterrência, a próxima etapa precisa de uma série `subprefeitura × mês × câmeras ativas` e de um desenho longitudinal.